In [4]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import warnings
warnings.filterwarnings("ignore")

builder = (
    SparkSession.builder
    .appName("delta-minio-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.2.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-86192deb-eddf-483a-b523-37408fe17835;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 82ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0

In [1]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

!spark-submit \
  --packages io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262 \
  --conf "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension" \
  --conf "spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog" \
  /workspace/rltm_bi_pltfrm/jobs/bronze_to_silver/bronze_to_silver.py

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-23d3d96d-dc5c-4600-8f91-bb7744af3725;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 150ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 f

In [8]:
silver_path = "s3a://lakehouse/silver/gold_price_ticks_clean"

df = spark.read.format("delta").load(silver_path)

print("row_count =", df.count())
df.printSchema()

row_count = 9
root
 |-- event_id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- source_name: string (nullable = true)
 |-- event_ts_utc: timestamp (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- price_usd: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- quality_flag: string (nullable = true)
 |-- processing_date: date (nullable = true)



In [9]:
df.select(
    "event_id",
    "symbol",
    "source_name",
    "event_ts_utc",
    "ingestion_ts",
    "price_usd",
    "currency",
    "quality_flag",
    "processing_date"
).orderBy("ingestion_ts", ascending=False).show(20, truncate=False)

+----------------------------------------------------------------+------+------------+-------------------+--------------------------+-----------+--------+------------+---------------+
|event_id                                                        |symbol|source_name |event_ts_utc       |ingestion_ts              |price_usd  |currency|quality_flag|processing_date|
+----------------------------------------------------------------+------+------------+-------------------+--------------------------+-----------+--------+------------+---------------+
|4b491db8309c4ddddac4e42a44bbfd39313c8d4c1ef4ad3568a17f535da12971|XAU   |gold_api_com|2026-03-21 17:31:10|2026-03-21 17:31:13.580731|4492.200195|USD     |OK          |2026-03-21     |
|e7ac851c1bd23e8de0a2e8d0c33f47144464957fbcea23bf4a284fdea0fedf1c|XAU   |gold_api_com|2026-03-21 17:18:10|2026-03-21 17:18:55.8226  |4492.200195|USD     |OK          |2026-03-21     |
|e0f2b8b42c7d23e3ee8ffcc641e363e839a17299ef773f1b537f9a9a31986fc0|XAU   |gold_ap